# Vector stores and semantic search



In [1]:
from sentence_transformers import SentenceTransformer

c:\Users\oscar\repos\clases-cetys\computational-intelligence\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Part I: Basic vector store implementation

In [2]:
import torch

class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata

    def __repr__(self):
        return f"Document(text={self.text}, metadata={self.metadata})"
        

class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document

    def __repr__(self):
        return f"Text: {self.document.text}\nScore: {self.score}"


class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.documents = []
        self.embeddings = None
        self.model = embedding_model

    def add_documents(self, documents: list[Document]):
        new_embeddings = self.model.encode(
            [doc.text for doc in documents],
            convert_to_tensor=True,
            normalize_embeddings=True,
        )

        if self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            self.embeddings = torch.cat([self.embeddings, new_embeddings], dim=0)

        self.documents.extend(documents)

    def get_n_documents(self, n: int) -> list[Document]:
        return self.documents[:n]

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        embeded_query = self.model.encode(query, convert_to_tensor=True, normalize_embeddings=True)
        scores = embeded_query @ self.embeddings.T

        sorted_scores, sorted_idx = torch.sort(scores, descending=True)
        top_scores = sorted_scores[:top_k]
        top_indices = sorted_idx[:top_k]

        return [SearchResult(score.item(), self.documents[idx]) for score, idx in zip(top_scores, top_indices)]

### Load dataset

In [3]:
import csv

In [4]:
documents = []

with open('data/animal-fun-facts-dataset.csv', mode='r', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    for i, row in enumerate(reader):
        text = str(row.pop('text')) 
        documents.append(Document(text=text, metadata=row))

print(f"Loaded {len(documents)} documents.\n")

for i, doc in enumerate(documents):
    if (i >= 5):
        break
    print(f"Document {i+1}:")
    print(doc.text)
    print(doc.metadata)
    print()


Loaded 7734 documents.

Document 1:
Aardvarks are sometimes called "ant bears", "earth pigs",
and "cape anteaters"
{'animal_name': 'aardvark', 'source': 'https://www.animalfactsencyclopedia.com/Aardvark-facts.html', 'media_link': '', 'wikipedia_link': '/wiki/Aardvark'}

Document 2:
Aardvarks
have rather primitive brains that are very small for the size of the
animal. Some have suggested they are not particularly bright....
{'animal_name': 'aardvark', 'source': 'https://www.animalfactsencyclopedia.com/Aardvark-facts.html', 'media_link': '', 'wikipedia_link': '/wiki/Aardvark'}

Document 3:
Aardvarks
teeth are lined with fine upright tubes and have no roots or enamel.
{'animal_name': 'aardvark', 'source': 'https://www.animalfactsencyclopedia.com/Aardvark-facts.html', 'media_link': '', 'wikipedia_link': '/wiki/Aardvark'}

Document 4:
The aardvarks Latin family name "Tubulidentata" means "tube toothed"
{'animal_name': 'aardvark', 'source': 'https://www.animalfactsencyclopedia.com/Aardvark-f

### Initialize Vector Store

In [5]:
model = SentenceTransformer("all-MiniLM-L6-v2")
store = VectorStore(model)
store.add_documents(documents)
print(model.device)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2786.97it/s]


cuda:0


In [6]:
print(store.get_n_documents(3))

[Document(text=Aardvarks are sometimes called "ant bears", "earth pigs",
and "cape anteaters", metadata={'animal_name': 'aardvark', 'source': 'https://www.animalfactsencyclopedia.com/Aardvark-facts.html', 'media_link': '', 'wikipedia_link': '/wiki/Aardvark'}), Document(text=Aardvarks
have rather primitive brains that are very small for the size of the
animal. Some have suggested they are not particularly bright...., metadata={'animal_name': 'aardvark', 'source': 'https://www.animalfactsencyclopedia.com/Aardvark-facts.html', 'media_link': '', 'wikipedia_link': '/wiki/Aardvark'}), Document(text=Aardvarks
teeth are lined with fine upright tubes and have no roots or enamel., metadata={'animal_name': 'aardvark', 'source': 'https://www.animalfactsencyclopedia.com/Aardvark-facts.html', 'media_link': '', 'wikipedia_link': '/wiki/Aardvark'})]


### Making queries

In [7]:
store.search("What is the fastest animal?")

[Text: Fastest animal on Earth
 Score: 0.9126603603363037,
 Text: The fastest creatures on the planet!
 Score: 0.8299250602722168,
 Text: The fastest land mammal in the world!
 Score: 0.8021312952041626,
 Text: The fastest species of primate in the world!
 Score: 0.7151623368263245,
 Text: Patagonian mara are the worlds fastest rodent
 .
 On that topic, they can really move! Plains animals often use speed to their advantage (think cheetahs and gazelles), but the mara is a record-holder in that department.
 Score: 0.7100554704666138]

In [8]:
store.search("Are there any animals that can fly?")

[Text: They don’t fly, they glide.
 The only mammal which can independently fly is the bat. Instead, colugas glide which works in the same way as a wingsuit.
 Score: 0.6900345087051392,
 Text: Not all birds are able to fly!
 Score: 0.6892896890640259,
 Text: Bats are the only mammals with wings, and the only ones that can truly fly
 Score: 0.6677284836769104,
 Text: They rarely fly..
 They move around on foot most of the time, only taking to the air to reach their nests or for courtship displays.
 Score: 0.6263091564178467,
 Text: Bats are the world's only flying mammals. Other mammals may glide through the air, but bats flap their wings and fly.
 Score: 0.6174590587615967]

In [9]:
store.search("Are golden retrievers good pets?")

[Text: Golden Pyrenees make great therapy dogs due to their intelligence and gentle nature.
 Score: 0.6628046035766602,
 Text: They are not great pets.
 In the USA and some other parts of the world they are kept as pets, however they are extremely strong and can be very dangerous as they get older. They are wild animals, and not domesticated.
 Score: 0.5993548631668091,
 Text: These dogs are very intelligent and are great with children.
 Score: 0.5695565938949585,
 Text: Although their coats can get incredibly light in color, golden retrievers never have purely white coats.
 Score: 0.565988302230835,
 Text: The Golden Shepherds were first recognized by the International Designer Canine Registry in 2009.
 Score: 0.5427731275558472]

In [10]:
store.search("Are crabs immortal?")

[Text: It may have the longest live span among crabs.
 Commonly known crabs—such as Dungeness crabs, king crabs, and snow crabs—live for several decades (between 10 to 30 years).
 Score: 0.7234090566635132,
 Text: There is a limit as to how large their bodies can grow.
 The carapace, or hard upper shell, of Japanese spider crabs cap at a particular size once it reaches adulthood. Their legs, however, keep on growing and elongating.
 Score: 0.5835776925086975,
 Text: Closely related to crabs and lobsters!
 Score: 0.5463244915008545,
 Text: Closely related to crabs and lobsters!
 Score: 0.5463244915008545,
 Text: Hermit crabs will eventually outgrow their shells. At such times, they'd line up according to size to swap shells. Shells too big or too small are rejected. This chain reaction is called "vacancy chain." Vacancy chain is a way for these crustaceans to survive while sharing limited resources.
 Score: 0.5435934662818909]

In [11]:
store.search("Hay un animal que hable español?")

[Text: They’re named after their hands.
 Both the Quebeqi and the Spanish-speaking colonists who named raccoons independently referred to the hands of the animal. The word aroughcoune roughly translates to “the one that scratches with his hands”, and mapachtli in Spanish refers to “the animal with the hands”.
 Score: 0.4280066192150116,
 Text: The resplendent quetzal is a sacred symbol in Mesoamerica and Guatemala's national bird, pictured on the country's flag. They favor eating fruit in the avocado family, eating them whole before regurgitating the pits. Essentially making them the avocado "gardeners" of their forest habitats.
 Score: 0.41232359409332275,
 Text: Axolotl is a Mexican word that translates to "Water dog"
 Score: 0.4037732481956482,
 Text: A hairless breed of dog!
 Score: 0.39489513635635376,
 Text: The second largest animal on the land!
 Score: 0.39259788393974304]

## Part II: Filtering by metadata

In [34]:
class FilteredSearchResult(SearchResult):
    def __init__(self, score: float, document: Document):
        super().__init__(score, document)
    
    def __repr__(self):
        return f"Text: {self.document.text}\nMetadata: {self.document.metadata}\nScore: {self.score}"

class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.documents = []
        self.embeddings = None
        self.model = embedding_model
    
    def add_documents(self, documents: list[Document]):
        new_embeddings = self.model.encode(
            [doc.text for doc in documents],
            convert_to_tensor=True,
            normalize_embeddings=True,
        )

        if self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            self.embeddings = torch.cat([self.embeddings, new_embeddings], dim=0)

        self.documents.extend(documents)


    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[FilteredSearchResult]:
        
        if metadata_filter:
            filtered_indices = []
            for idx, doc in enumerate(self.documents):
                if any(doc.metadata.get(key) == value for key, value in metadata_filter.items()):
                    filtered_indices.append(idx)

            if not filtered_indices:
                return []

            filtered_embeddings = self.embeddings[filtered_indices]

        embeded_query = self.model.encode(query, convert_to_tensor=True, normalize_embeddings=True)

        if metadata_filter:
            scores = embeded_query @ filtered_embeddings.T
        else:
            scores = embeded_query @ self.embeddings.T

        sorted_scores, sorted_idx = torch.sort(scores, descending=True)
        top_scores = sorted_scores[:top_k]
        top_indices = sorted_idx[:top_k]

        return [FilteredSearchResult(score.item(), self.documents[filtered_indices[idx]]) for score, idx in zip(top_scores, top_indices)]

In [35]:
model = SentenceTransformer("all-MiniLM-L6-v2")
filtered_store = FilteredVectorStore(model)
filtered_store.add_documents(documents)
print(model.device)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6418.93it/s]


cuda:0


Test with the animal fun facts dataset

In [37]:
filtered_store.search("Carnivorous", metadata_filter={"animal_name": "lion"})

[Text: The lion diet is primarily meat..
 African lions hunt and consume large animals they find in the grasslands they inhabit. These animals include zebras, wildebeest and antelopes. The male lion requires 7-kilograms of food daily and the female can survive on 5-kilos.
 Metadata: {'animal_name': 'lion', 'source': 'https://factanimal.com/lions/', 'media_link': '', 'wikipedia_link': '/wiki/Lion'}
 Score: 0.36430808901786804,
 Text: Like cats, lions have sharp spines on tongue called papillae. Those spines have a hollow cavity at the tip, which can contain saliva. Lion can use papillae to spread saliva on fur, reducing body temperature by evaporation.
 Metadata: {'animal_name': 'lion', 'source': 'https://reddit.com/r/Awwducational/comments/whr66y/like_cats_lions_have_sharp_spines_on_tongue/', 'media_link': 'https://i.redd.it/pg39u4s254g91.jpg', 'wikipedia_link': '/wiki/Lion'}
 Score: 0.3408797085285187,
 Text: African lions live in groups called ‘prides’..
 The African lion is actually

In [41]:
filtered_store.search("Carnivorous", metadata_filter={"animal_name": "leopard"})

[Text: Leopards can drag prey twice their size high up into trees to keep them from scavengers
 Metadata: {'animal_name': 'leopard', 'source': 'https://www.animalfactsencyclopedia.com/Leopard-facts.html', 'media_link': '', 'wikipedia_link': '/wiki/Leopard'}
 Score: 0.3991592824459076,
 Text: They rarely attack humans, but it does happen.
 Leopards are the least likely of the big cats to become man eaters, although there are around 55 humans killed in India each year where humans come into contact with leopard populations.
 Metadata: {'animal_name': 'leopard', 'source': 'https://factanimal.com/leopard/', 'media_link': '', 'wikipedia_link': '/wiki/Leopard'}
 Score: 0.3808819055557251,
 Text: Leopards hang their kill in trees.
 Metadata: {'animal_name': 'leopard', 'source': 'https://www.animalfactsencyclopedia.com/Leopard-facts.html', 'media_link': '', 'wikipedia_link': '/wiki/Leopard'}
 Score: 0.36025679111480713,
 Text: While rare, tigers and lions can kill and consume leopards.
 Inters

### Load the dataset

This dataset contains metadata for the Top 10,000 movies currently listed on The Movie Database (TMDB). The data captures the highest-rated films globally, filtered by those that have met a minimum vote threshold to ensure quality and relevance.

The purpose of using this dataset is to find movies based on their overview.

The dataset is sourced from Kaggle: https://www.kaggle.com/datasets/rosemeenshaikh/top-10000-tmdbs-highest-rated-movies

In [55]:
# Since the movie dataset does not have a lot of categorical metadata, we will create some based on the release date and vote average.

rows = []
with open("data/top_rated_movies.csv", mode="r", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    for row in reader:
        row["vote_average"] = float(row["vote_average"])
        rows.append(row)

votes = [r["vote_average"] for r in rows]
avg_vote = sum(votes) / len(votes)

documents = []
for row in rows:
    overview = row.pop("overview")

    # Century flag from release_date (YYYY-MM-DD)
    release_date = row.get("release_date", "")
    
    try:
        year = int(release_date[:4]) if release_date else None
    except ValueError:
        year = None

    if year is None:
        century = "unknown"
    elif year <= 2000:
        century = "20"
    else:
        century = "21"

    row["century"] = century
    row["above_avg"] = "True" if (row["vote_average"] is not None and row["vote_average"] >= avg_vote) else "False"

    documents.append(Document(text=overview, metadata=row))

print(f"Loaded {len(documents)} documents.\n")
print(documents[0:5])

Loaded 10000 documents.

[Document(text=Imprisoned in the 1940s for the double murder of his wife and her lover, upstanding banker Andy Dufresne begins a new life at the Shawshank prison, where he puts his accounting skills to work for an amoral warden. During his long stretch in prison, Dufresne comes to be admired by the other inmates -- including an older prisoner named Red -- for his integrity and unquenchable sense of hope., metadata={'adult': 'False', 'id': '278', 'original_language': 'en', 'original_title': 'The Shawshank Redemption', 'popularity': '46.3708', 'release_date': '1994-09-23', 'title': 'The Shawshank Redemption', 'vote_average': 8.718, 'vote_count': '30171', 'century': '20', 'above_avg': 'True'}), Document(text=Spanning the years 1945 to 1955, a chronicle of the fictional Italian-American Corleone crime family. When organized crime family patriarch, Vito Corleone barely survives an attempt on his life, his youngest son, Michael steps in to take care of the would-be k

### Initializing the vector store

In [56]:
model = SentenceTransformer("all-MiniLM-L6-v2")
filtered_store = FilteredVectorStore(model)
filtered_store.add_documents(documents)
print(model.device)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6211.82it/s]


cuda:0


### Making queries with filters

In [59]:
filtered_store.search("Mafia and crime", metadata_filter={"original_language": "en"})

[Text: Two of New York's most notorious organized crime bosses, Frank Costello and Vito Genovese, vie for control of the city's streets. Once the best of friends, petty jealousies and a series of betrayals place them on a deadly collision course that will reshape the Mafia (and America) forever.
 Metadata: {'adult': 'False', 'id': '1013601', 'original_language': 'en', 'original_title': 'The Alto Knights', 'popularity': '3.8274', 'release_date': '2025-03-19', 'title': 'The Alto Knights', 'vote_average': 6.2, 'vote_count': '415', 'century': '21', 'above_avg': 'False'}
 Score: 0.6186681985855103,
 Text: To take down South Boston's Irish Mafia, the police send in one of their own to infiltrate the underworld, not realizing the syndicate has done likewise. While an undercover cop curries favor with the mob kingpin, a career criminal rises through the police ranks. But both sides soon discover there's a mole among them.
 Metadata: {'adult': 'False', 'id': '1422', 'original_language': 'en', '

In [58]:
filtered_store.search("Mafia and crime", metadata_filter={"above_avg": "True"})

[Text: To take down South Boston's Irish Mafia, the police send in one of their own to infiltrate the underworld, not realizing the syndicate has done likewise. While an undercover cop curries favor with the mob kingpin, a career criminal rises through the police ranks. But both sides soon discover there's a mole among them.
 Metadata: {'adult': 'False', 'id': '1422', 'original_language': 'en', 'original_title': 'The Departed', 'popularity': '13.4085', 'release_date': '2006-10-04', 'title': 'The Departed', 'vote_average': 8.16, 'vote_count': '16081', 'century': '21', 'above_avg': 'True'}
 Score: 0.5817553400993347,
 Text: An FBI undercover agent infiltrates the mob and identifies more with the mafia life at the expense of his regular one.
 Metadata: {'adult': 'False', 'id': '9366', 'original_language': 'en', 'original_title': 'Donnie Brasco', 'popularity': '8.0643', 'release_date': '1997-02-27', 'title': 'Donnie Brasco', 'vote_average': 7.538, 'vote_count': '4820', 'century': '20', 'ab

In [60]:
filtered_store.search("Soccer", metadata_filter={"century": "21"})

[Text: Madrid, Spain, 2010. While the whole city follows the national team's successful participation in the World Cup, a group of daring thieves look for a way into one of the most secure and guarded places on the planet.
 Metadata: {'adult': 'False', 'id': '630004', 'original_language': 'en', 'original_title': 'Way Down', 'popularity': '4.293', 'release_date': '2021-03-04', 'title': 'The Vault', 'vote_average': 6.804, 'vote_count': '1060', 'century': '21', 'above_avg': 'True'}
 Score: 0.44361957907676697,
 Text: A light hearted comedy about the beginnings of Professional American Football. When a decorated war hero and college all star is tempted into playing professional football. Everyone see the chance to make some big money, but when a reporter digs up some dirt on the war hero... everyone could lose out.
 Metadata: {'adult': 'False', 'id': '4942', 'original_language': 'en', 'original_title': 'Leatherheads', 'popularity': '2.6325', 'release_date': '2008-03-24', 'title': 'Leatherh

In [61]:
filtered_store.search("Soccer", metadata_filter={"century": "20"})

[Text: A football player and his mates travel to the planet Mongo and find themselves fighting the tyranny of Ming the Merciless to save Earth.
 Metadata: {'adult': 'False', 'id': '3604', 'original_language': 'en', 'original_title': 'Flash Gordon', 'popularity': '5.7205', 'release_date': '1980-12-05', 'title': 'Flash Gordon', 'vote_average': 6.234, 'vote_count': '1080', 'century': '20', 'above_avg': 'False'}
 Score: 0.43082982301712036,
 Text: In a corporate-controlled future, an ultra-violent sport known as Rollerball represents the world, and one of its powerful athletes is out to defy those who want him out of the game.
 Metadata: {'adult': 'False', 'id': '11484', 'original_language': 'en', 'original_title': 'Rollerball', 'popularity': '2.632', 'release_date': '1975-06-25', 'title': 'Rollerball', 'vote_average': 6.293, 'vote_count': '600', 'century': '20', 'above_avg': 'False'}
 Score: 0.372898131608963,
 Text: In a boorish future, the government sponsors a popular, but bloody, cros

In [64]:
filtered_store.search("Mexican", metadata_filter={"above_avg": "True"})

[Text: In Monterrey, Mexico, a young street gang spends their days dancing to slowed-down cumbia and attending parties. After a mix-up with a local cartel, their leader is forced to migrate to the U.S. but quickly longs to return home.
 Metadata: {'adult': 'False', 'id': '582927', 'original_language': 'es', 'original_title': 'Ya no estoy aquí', 'popularity': '0.9082', 'release_date': '2019-10-21', 'title': "I'm No Longer Here", 'vote_average': 7.871, 'vote_count': '524', 'century': '21', 'above_avg': 'True'}
 Score: 0.444307804107666,
 Text: In this biographical drama, Selena Quintanilla is born into a musical Mexican-American family in Texas. Her father, Abraham, realizes that his young daughter is talented and begins performing with her at small venues. She finds success and falls for her guitarist, Chris Perez, who draws the ire of her father. Seeking mainstream stardom, Selena begins recording an English-language album which, tragically, she would never complete.
 Metadata: {'adult

## Reflections

This activity helped me understand how vector stores work, how embeddings represent meaning, why cosine similarity retrieves relevant vectors, how metadata filtering sharpens results, and how RAG can be built on top of these ideas.